# 26 · Apprentissage par renforcement — solutions (notebook exécuté)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookIA/blob/main/chapitres/ch26_rl/05_solutions.ipynb)

Les réponses des exercices, exécutées. Les démarches détaillées (le *pourquoi*, les erreurs fréquentes, les variantes) sont dans `05_solutions.md`. Les cellules marquées `answer` enregistrent les réponses vérifiées par `wb.check` (`tools/build_answers.py`).

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | 26.Q1 à 26.Q12, 26.R1 à 26.R3, 26.1 à 26.6 | vérifier tes réponses courtes | 🧠 🔁 ✏️ | ★ à ★★ | |
| A | 26.13 à 26.18 | la boucle et Gymnasium, les retours, Flippers et le L-learning (*prochaine session de génération*) | | | |
| B | 26.19 à 26.21 | le cœur de ta librairie `rl.py` : politiques, mises à jour, boucle d'entraînement (*prochaine session de génération*) | | | |
| C | 26.22 à 26.27 | le Q-learning en action : FrozenLake, courbes, un 🐛, Flippers bruité, itération sur les valeurs, hyperparamètres (*prochaine session de génération*) | | | |
| D | 26.28 à 26.30 | SARSA contre le Q-learning, au bord de la falaise et sur Flippers (*prochaine session de génération*) | | | |
| E | 26.31 à 26.34 | le rejeu d'expérience, la conception des récompenses, un résultat honnête (*prochaine session de génération*) | | | |
| F | 26.35 à 26.38 | le morpion, CartPole, l'auto-apprentissage et le défi (*prochaine session de génération*) | | | |

In [1]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (CPU, a few minutes); False = full version
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookIA.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookIA")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WB_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wb").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wb").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier du workbook.")

os.environ["WB_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wb  # noqa: E402

cfg = wb.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mylearn = wb.load_mylearn("ref")  # reference implementation

🧪 Deep Learning Workbook : environnement
   Plateforme : Local (Linux x86_64, 2 CPU)
   Python 3.13.18 | numpy 2.1.3 | pandas 2.2.3 | matplotlib 3.10.0 | scikit-learn 1.6.1 | scipy 1.16.3 | torch 2.11.0+cpu | torchvision 0.26.0+cpu
   Device : cpu | Graine : 42 | FAST_MODE : True


## Objectifs et rappel express

- Manipuler la boucle agent-environnement avec Gymnasium, et calculer des retours actualisés.
- Écrire dans `mylearn.rl` un agent tabulaire complet (politiques, mises à jour du Q-learning et de SARSA, boucle d'entraînement, itération sur les valeurs, mémoire de rejeu) et le comparer sur Flippers, FrozenLake, CliffWalking, le morpion et CartPole.
- Rapporter un résultat de RL sur plusieurs graines.

**Rappel express.** $G_t = r_t + \gamma\,G_{t+1}$. Q-learning : $Q(s, a) \leftarrow Q(s, a) + \alpha\,[r + \gamma \max_{a'} Q(s', a') - Q(s, a)]$, la cible vaut $r$ si $s'$ est terminal. SARSA : $Q(s', a')$ de l'action réellement choisie au lieu du max. ε-greedy : l'action gloutonne avec la probabilité $1 - \varepsilon + \varepsilon/|\mathcal{A}|$.

## Partie 0 · Vérifier tes réponses courtes (quiz, rappels, ✏️ 26.1 à 26.6)

Fais d'abord les quiz, les rappels et les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse courte ici : **la valeur** que tu as trouvée, pas l'expression Python, sinon tu ne vérifies rien. Un nombre : `42` ou `0.3` (en Python, le séparateur décimal est un **point** ; `0,3` sans guillemets serait un couple de deux nombres) ; un vrai ou faux : `True` ou `False` ; un choix : la lettre seule, entre guillemets (`"F"`) ; plusieurs lettres : entre guillemets (`"AF"`), dans l'ordre demandé quand l'énoncé en impose un. Arrondis comme l'énoncé le demande, et seulement à la fin du calcul. Les réponses pas encore remplies affichent ⏳. Les questions « dans ta copie », les dérivations ∂ 26.7 et 26.8, l'estimation 🧮 26.9, la réflexion (🗣️ 26.10, ⚖️ 26.11, 📄 26.12) et l'entretien se corrigent avec `05_solutions.md`.

In [2]:
# The short answers only use the numbers of their statements (02_exercices.md)
import math


def discounted(rewards, gamma):
    """G_0 = sum of gamma**k * r_k."""
    return sum(gamma ** k * r for k, r in enumerate(rewards))


def softmax_prob(q, tau, i):
    """Probability of action i under a softmax of temperature tau."""
    m = max(q)
    weights = [math.exp((v - m) / tau) for v in q]
    return weights[i] / sum(weights)


REWARDS_26_1 = [1, 0, 0, 2, 0, 4]
Q_26_3, R_26_3, NEXT_26_3, GAMMA_26_3, ALPHA_26_3 = 0.5, 0.2, [0.1, 0.6, 0.3, 0.6], 0.9, 0.5
TARGET_26_3 = R_26_3 + GAMMA_26_3 * max(NEXT_26_3)
NEW_26_3 = Q_26_3 + ALPHA_26_3 * (TARGET_26_3 - Q_26_3)
TARGET_26_4 = R_26_3 + GAMMA_26_3 * NEXT_26_3[2]
NEW_26_4 = Q_26_3 + ALPHA_26_3 * (TARGET_26_4 - Q_26_3)
Q_26_5, EPS_26_5 = [1, 2, 4, 0], 0.2


def chain_26_6(alpha, episodes, n_states=3, gamma=0.9):
    """Q of the chain 0 -> 1 -> ... -> n_states (terminal), updated in the order of the steps."""
    q = [0.0] * n_states
    for _ in range(episodes):
        for s in range(n_states):
            target = 1.0 if s == n_states - 1 else gamma * q[s + 1]
            q[s] += alpha * (target - q[s])
    return q

**26.Q1 — Apprendre avec des récompenses : ni supervisé, ni non supervisé**

In [3]:
wb.record("26.Q1a", "B", mistakes={"le RL utilise souvent des réseaux : relis ce que l'agent reçoit à la place d'un label": "A", "un agent apprend de ses propres essais : ce sont des données": "C", "relis les exemples du livre : ascenseurs, cultures, robot": "D"})
wb.record("26.Q1b", "ACE", mistakes={"un des problèmes cités se résout avec des labels, un autre sans aucun signal : ni l'un ni l'autre n'est du RL": "ABCE",
          "il manque un problème où l'on agit et où l'on est jugé sur le résultat": "AC",
          "le regroupement sans labels est de l'apprentissage non supervisé": "ACDE"})
wb.record("26.Q1c", False, mistakes={"relis le §26.2 : quand arrive la récompense d'une partie gagnée ?": True})
wb.record("26.Q1d", False, mistakes={"relis le §26.1 : le livre parle d'une approche générale": True})
wb.record("26.Q1e", "B", mistakes={"le livre dit qu'il n'existe souvent pas de réponse objectivement juste": "A", "relis le §26.2 : le critère est celui qu'on choisit": "C"})

WB_ANSWER {"hash": "3b16861541defdb5922ebc7b1ea45220a532c31954db75207b140e32fcd7ff58", "id": "26.Q1a", "kind": "str", "mistakes": {"b3ab87506659d49382924252d3b87c864ef9321f3fd048d87a64b3717c59ba5e": "le RL utilise souvent des réseaux : relis ce que l'agent reçoit à la place d'un label", "e7a01a2fcf29b6d23096e137a380649ed4eab3cdaceb1f23c8bd16997046c3e1": "un agent apprend de ses propres essais : ce sont des données", "f980ee6b3cb789a11bbae3d361c9d575f1f6cb70d583ec9812e4b27de13a2062": "relis les exemples du livre : ascenseurs, cultures, robot"}}
📝 Ex 26.Q1a : réponse enregistrée (str).
WB_ANSWER {"hash": "9e595933637d0e91b3b46bd76f67917780295b2ad846df03491e39ffb86078ca", "id": "26.Q1b", "kind": "str", "mistakes": {"08a8bcbfc1d1dae69d7c7becdf01fecbb1ff8f3b09653923d9191780eb41fe0f": "le regroupement sans labels est de l'apprentissage non supervisé", "77be7d2fe95e1621e6b7913ac9655b03a8e9b19dd96177341b3817cbd80f4bf0": "un des problèmes cités se résout avec des labels, un autre sans aucun sig

**26.Q2 — Les quatre informations données à l'agent du morpion**

In [4]:
wb.record("26.Q2a", "B", mistakes={"l'amie choisit les coups : c'est l'agent": "A", "relis qui donne les récompenses et joue en retour": "C"})
wb.record("26.Q2b", "ABDF", mistakes={"l'agent ne reçoit pas la règle du jeu : il l'apprend": "ABCDF",
          "il manque une des quatre informations": "ABD",
          "on ne lui dit même pas que c'est un jeu": "ABDEF"})
wb.record("26.Q2c", False, mistakes={"relis ce que le livre dit des informations privées de l'agent": True})
wb.record("26.Q2d", "B", mistakes={"proposer des coups interdits ne fait pas gagner de temps": "A", "relis le §26.2.1 : à quoi servent les récompenses d'un coup interdit ?": "C"})
wb.record("26.Q2e", 9 - 2, mistakes={"les coups permis sont les cases vides": 2})

WB_ANSWER {"hash": "cae1e6c47800539e0d4537e2c6a5f45d93fffe5ee385408ffd77088cb1941fe1", "id": "26.Q2a", "kind": "str", "mistakes": {"67c99b35f744f20ba53d1624d0d1ae719dd82842c1d7eccb4da965c5c38bc3a5": "l'amie choisit les coups : c'est l'agent", "b47c1f281806111d5f73582d939c4e925f8e75167a995fa2c831351c65d17850": "relis qui donne les récompenses et joue en retour"}}
📝 Ex 26.Q2a : réponse enregistrée (str).
WB_ANSWER {"hash": "92ea1d9e4aaa437eb06b32680c73f305b62f20bc147290e20e615e2f22586aa2", "id": "26.Q2b", "kind": "str", "mistakes": {"098b2c96e9b8614fb0157075ec3ad0b85bab5c97bc35ce2016e4433e79fac7a4": "l'agent ne reçoit pas la règle du jeu : il l'apprend", "0a092cbe8824e4e361124cbabe3347b9eef727dcecc6443bda39157439c1e134": "on ne lui dit même pas que c'est un jeu", "43c73e12878a3f6ab4ca9d7c4e33772ea928b5b4883ac2bc63ea1388efcefb6c": "il manque une des quatre informations"}}
📝 Ex 26.Q2b : réponse enregistrée (str).
WB_ANSWER {"hash": "36b02c79e58855cee76dff7e1c459ea6bcf76f24293ed833b9ecd783f

**26.Q3 — État, actions, politique et informations privées**

In [5]:
wb.record("26.Q3a", "CAB", mistakes={"l'agent agit avant que l'environnement réponde": "ACB",
          "l'agent se met à jour après la réponse": "CBA"})
wb.record("26.Q3b", "B", mistakes={"une epoch est un passage sur un dataset, pas une partie": "A", "la politique est la règle de choix des actions": "C", "une transition est le passage d'un état à un autre": "D"})
wb.record("26.Q3c", "B", mistakes={"une moyenne de récompenses mesure un résultat, pas une règle de choix": "A", "la liste des états décrit le monde, pas ce qu'on y fait": "C", "le plateau de départ est un état, pas une règle": "D"})
wb.record("26.Q3d", False, mistakes={"relis le §26.3.1 : qui exécute l'action, et pourquoi ?": True})
wb.record("26.Q3e", True, mistakes={"relis le §26.3.1 : un état contient tous les nombres qui décrivent l'environnement": False})

WB_ANSWER {"hash": "123574aeb60640a0b6431fc186745d73b63bb9ed56231e3df7ce5494b129e32a", "id": "26.Q3a", "kind": "str", "mistakes": {"3a4312d2cbaa8ea0ed0396f6c06b051e9b2363379a392dde6a47bd45cbe4504a": "l'agent agit avant que l'environnement réponde", "935969b5861cbd5c5054a552fb5d351c925780f4673452d5309402d727431dea": "l'agent se met à jour après la réponse"}}
📝 Ex 26.Q3a : réponse enregistrée (str).
WB_ANSWER {"hash": "1cdf5ad1580feb66e44f4813a39ac9e391f511d8555b9e4f4e7faf089c74f493", "id": "26.Q3b", "kind": "str", "mistakes": {"488f60616a3ee1ea8f13e6a6028c2167044334bf2963dd76c668ac4a872003d0": "une transition est le passage d'un état à un autre", "5b56ce066f4fcb7d1a8b8052929981649245ed7584424b0b42aec5bbf629b6f8": "une epoch est un passage sur un dataset, pas une partie", "842fa3fc41feb81c08844d18e9eadd2522fc3bcbf0d3a0add914a816cdd44ea5": "la politique est la règle de choix des actions"}}
📝 Ex 26.Q3b : réponse enregistrée (str).
WB_ANSWER {"hash": "7572e1344c7a932771d1a361699409b2cbdb4f7

**26.Q4 — Réponse de l'environnement, épisode et agents libres ou déclenchés**

In [6]:
wb.record("26.Q4a", "B", mistakes={"la politique appartient à l'agent": "A", "l'environnement ne connaît pas la meilleure action": "C", "les valeurs Q sont calculées par l'agent": "D"})
wb.record("26.Q4b", "BA", mistakes={"le cuisinier attend la sonnerie : il est déclenché ; le minuteur tourne seul": "AB",
          "le minuteur tourne de lui-même": "BB",
          "le cuisinier attend qu'on le sollicite": "AA"})
wb.record("26.Q4c", "B", mistakes={"au backgammon, chacun attend son tour": "A", "les deux joueurs attendent leur tour": "C"})
wb.record("26.Q4d", True, mistakes={"relis le §26.3.4 : lequel des quatre cas le livre dit-il le plus simple ?": False})
wb.record("26.Q4e", "C", mistakes={"il manque les indicateurs de fin et le dictionnaire info": "A", "c'est ce que renvoie reset, pas step": "B", "c'est l'ancienne API de Gym : Gymnasium sépare deux façons de finir": "D"})

WB_ANSWER {"hash": "ae500f407936114398e54c1e5c7ce982cef503445786b07a45050136217df561", "id": "26.Q4a", "kind": "str", "mistakes": {"3026bb0646a540533a10ee337f064ac1e129928763860b11ec59eeb5df823dc9": "la politique appartient à l'agent", "c33bea5131431d13c9377d42f61d9448568d48ed39ba067651b303b2bef747ec": "l'environnement ne connaît pas la meilleure action", "db5f3bd2db198cced45d5b54d54b11ce5f6ad3b286e8e561d571f51ae72da104": "les valeurs Q sont calculées par l'agent"}}
📝 Ex 26.Q4a : réponse enregistrée (str).
WB_ANSWER {"hash": "081cdd5c94d2cd35919d46416e35b572206af28a0dd3ba1e7fb1b2afd0cb7b19", "id": "26.Q4b", "kind": "str", "mistakes": {"17943718359f89ae8588e198392500a267aa6cd3450d7c76604ebc8854173f9d": "le cuisinier attend la sonnerie : il est déclenché ; le minuteur tourne seul", "b960baec91376be196bfd729b72ffcb0f60ce772cfc2217ef62af44080506da2": "le cuisinier attend qu'on le sollicite", "fce2c3259425ae46ee5780e62791474eac071ee14816ee886f4f9ca4cd667d6c": "le minuteur tourne de lui-même

**26.Q5 — MDP, observabilité, attribution du crédit, exploration ou exploitation**

In [7]:
wb.record("26.Q5a", "B", mistakes={"relis l'encadré sur les MDP : il s'agit de mémoire, pas d'équiprobabilité": "A", "la propriété de Markov ne dit rien du signe des récompenses": "C", "un MDP peut être stochastique": "D"})
wb.record("26.Q5b", "B", mistakes={"l'agent voit-il tout l'état ?": "A"})
wb.record("26.Q5c", "B", mistakes={"le dilemme porte sur le choix du prochain coup, pas sur le mérite des coups passés": "A", "il s'agit de partager le mérite d'une victoire entre les coups": "C", "l'actualisation pèse les récompenses futures, elle ne dit pas quel coup a compté": "D"})
wb.record("26.Q5d", False, mistakes={"une politique gloutonne rejoue toujours la même action : une action sous-estimée n'est plus jamais essayée": True})
wb.record("26.Q5e", "B", mistakes={"la régression logistique ne choisit pas d'actions": "A", "le k-means regroupe des données, il ne choisit pas d'actions": "C"})

WB_ANSWER {"hash": "88a863c5b5f372aa8a03f2f25901f03470331b01db211469b169df83f1d21021", "id": "26.Q5a", "kind": "str", "mistakes": {"09bd1bd66a11a23c70e6499e0aeb595862ee9e477a74a153c9bb1a418dc9221e": "un MDP peut être stochastique", "11f584a6f2b0196ebcb1986f20e5e3390d56a90210194fe1d46c9fca4f38811c": "la propriété de Markov ne dit rien du signe des récompenses", "65171feae06e5d9b795564fb496aab13521cb05e59be91e5e9afdcdc6c2b3f32": "relis l'encadré sur les MDP : il s'agit de mémoire, pas d'équiprobabilité"}}
📝 Ex 26.Q5a : réponse enregistrée (str).
WB_ANSWER {"hash": "50aecb4c02974f2e72fa9b4d8f152902e38d5761404c6c5806156e69e7718b5f", "id": "26.Q5b", "kind": "str", "mistakes": {"440f890e5a41cb7ca3a49e84d4ae3480d292226ef639444cd09c8c3982c8ad53": "l'agent voit-il tout l'état ?"}}
📝 Ex 26.Q5b : réponse enregistrée (str).
WB_ANSWER {"hash": "e3e9d0dad7684dc63ef6e260481d94a8de493a785db0e8925db411c97d088679", "id": "26.Q5c", "kind": "str", "mistakes": {"34dc6a10bbefbc1ce4b9a6a3be8fb76dcc7c126c5126

**26.Q6 — Retour total, retour actualisé et rôle de γ**

In [8]:
wb.record("26.Q6a", 2 + 0 + 1)
wb.record("26.Q6b", discounted([2, 0, 1], 0.5), decimals=2, mistakes={"le poids de la 3e récompense est γ², pas γ": 2.5, "c'est le retour total : actualise": 3})
wb.record("26.Q6c", True, mistakes={"avec γ = 0, que valent les poids γ^k pour k ≥ 1 ?": False})
wb.record("26.Q6d", "B", mistakes={"c'est l'interprétation du livre, que la fiche corrige": "A", "c'est le rôle du learning rate α": "C"})
wb.record("26.Q6e", round(1 / (1 - 0.95)), mistakes={"1 / (1 − γ), pas γ / (1 − γ)": 19})

WB_ANSWER {"hash": "5cc09472abed251e357c44ac712f631ab3341e29166d4935ddf8df366033b614", "id": "26.Q6a", "kind": "int", "magnitude": 0, "sign": 1}
📝 Ex 26.Q6a : réponse enregistrée (int).
WB_ANSWER {"alt_coarse": ["67d5d00933a6eefbc7c2472d36cb82a0fbf428bcd030c4b70029f6cc6f382bf1"], "decimals": 2, "hash": "c6826b9e87bf39451ce2b2e7603285964971b54588b6236406a1f9cec5fd24e0", "hash_coarse": "b9d684fa8b4ab9f232dcae067449bb2f2abc25efe1bb7fddda9043a6dd6740f4", "id": "26.Q6b", "kind": "float", "magnitude": 0, "mistakes": {"2dd2abc7c40055fa0d98669de02dadbb87a664bd6b392dbd80ffb3b3af2bbae5": "c'est le retour total : actualise", "412cd84c156c1b955c2a64c87dc6050c9b3351f8bda0b2810f49b29f32889c6e": "le poids de la 3e récompense est γ², pas γ"}, "sign": 1}
📝 Ex 26.Q6b : réponse enregistrée (float, 2 décimale(s)).
WB_ANSWER {"hash": "c22f66f63bad161a5eeb9820a1e1a677df46de3dc4300da02799dce4c4519003", "id": "26.Q6c", "kind": "bool", "mistakes": {"d81cb38918035dae5cb5551e43f4133487c95c11ced7f3d48a9769a6c05f2

**26.Q7 — Flippers et L-learning : pourquoi « lousy » ?**

In [9]:
wb.record("26.Q7a", "B", mistakes={"relis la règle du jeu au §26.4": "A", "les diagonales ne comptent pas, et les autres tuiles doivent être vides": "C"})
wb.record("26.Q7b", 6, mistakes={"les diagonales ne comptent pas": 8, "les colonnes comptent aussi": 3})
wb.record("26.Q7c", "B", mistakes={"la moyenne, c'est justement ce qui manque au L-learning": "A", "relis la règle de mise à jour du §26.5": "C"})
wb.record("26.Q7d", True, mistakes={"relis les résultats du livre sans camion, après 6 000 parties": False})
wb.record("26.Q7e", "B", mistakes={"le jeu reste gagnable, seulement plus long": "A", "le camion retourne des tuiles, il n'en ajoute pas": "C"})

WB_ANSWER {"hash": "538df74ac7680383b792419d8b4ac0d4f88b2172aa5f8adfbcd8eb9cab660cf6", "id": "26.Q7a", "kind": "str", "mistakes": {"34304e5dda609ebedffeebec871b14a530d9e95ded64d7b2fc5d7ff4e81e6570": "relis la règle du jeu au §26.4", "e68e3d100455e2ab76bfafbc06d7c831aef828436decaaff2b7c0968026e4f7f": "les diagonales ne comptent pas, et les autres tuiles doivent être vides"}}
📝 Ex 26.Q7a : réponse enregistrée (str).
WB_ANSWER {"hash": "2bd5087d363a4f8a646373592a7f114829a42cc072284f596b483da485cfea88", "id": "26.Q7b", "kind": "int", "magnitude": 0, "mistakes": {"0ab4b1ac01bac0567e2c0924a1769a8b0a1ea3a221c1a7d4f8cb3d443f60d652": "les diagonales ne comptent pas", "539be6e9fd910fce109ab6bc7e461738b70b15f9a1163dfe515a1915f6297dfa": "les colonnes comptent aussi"}, "sign": 1}
📝 Ex 26.Q7b : réponse enregistrée (int).
WB_ANSWER {"hash": "18f0e36417c12a0252ce9d003b7be169ff531b3e91ec1f90b64bece65262d612", "id": "26.Q7c", "kind": "str", "mistakes": {"100da9b4fb3bdd1859015353a306ad35b97196756315f44d7

**26.Q8 — La mise à jour du Q-learning : α, γ et la valeur de l'état suivant**

In [10]:
wb.record("26.Q8a", "B", mistakes={"la ligne de s porte la valeur qu'on met à jour : la cible regarde ce qui vient après": "A", "la cible regarde l'état atteint par l'action, pas celui du départ": "C"})
wb.record("26.Q8b", "B", mistakes={"c'est le cas α = 1": "A", "α = 0 laisse l'ancienne valeur": "C"})
wb.record("26.Q8c", "B", mistakes={"la moyenne, c'est 0 < α < 1": "A", "c'est le cas α = 0": "C"})
wb.record("26.Q8d", False, mistakes={"relis le §26.6 : quand le Q-learning met-il la table à jour ?": True})
wb.record("26.Q8e", 0.4 + 0.5 * (0 + 1 * 0.8 - 0.4), decimals=2, mistakes={"c'est la cible : mélange-la avec l'ancienne valeur": 0.8})

WB_ANSWER {"hash": "1f91f4aae10cb34887cb635e08114355c04726d4b34e34790d27fa520648bf59", "id": "26.Q8a", "kind": "str", "mistakes": {"059792eb9d9cb257fac67d033fc81764f7e5d7750f4e1fa4993827a7d8253cf5": "la ligne de s porte la valeur qu'on met à jour : la cible regarde ce qui vient après", "e0c878f5194ccaf4d8d9129f1c863cf9b9424241dbf21f7f687cc786f5fd4c3b": "la cible regarde l'état atteint par l'action, pas celui du départ"}}
📝 Ex 26.Q8a : réponse enregistrée (str).
WB_ANSWER {"hash": "b09242176cd352570fd1f08c9665cb70ed1e2ba62f62f98f1495f41124c39ddc", "id": "26.Q8b", "kind": "str", "mistakes": {"51c63c6a732169c53e8114a3a6e912b4d6c18b2e36b1d4f5f563e3ea63603f93": "c'est le cas α = 1", "d7a74b016ba59bb25a246f85813a760d46d8012e735617bde291f6266cafdd2b": "α = 0 laisse l'ancienne valeur"}}
📝 Ex 26.Q8b : réponse enregistrée (str).
WB_ANSWER {"hash": "a6dd5e9d92a352d8175119c2771f0f27424e31b3d8d4d288114d70629e26db95", "id": "26.Q8c", "kind": "str", "mistakes": {"3f90807716064fcce6361b8727c68dac3ad0d

**26.Q9 — ε-greedy ou softmax : vrai ou faux**

In [11]:
wb.record("26.Q9a", True, mistakes={"c'est la version du livre que la fiche corrige": False})
wb.record("26.Q9b", 1 - 0.1 + 0.1 / 5, decimals=2, mistakes={"l'exploration peut aussi tirer l'action gloutonne": 0.9})
wb.record("26.Q9c", False, mistakes={"une grande température rend les probabilités presque égales": True})
wb.record("26.Q9d", True, mistakes={"relis la fin du paragraphe softmax du §26.6.2": False})
wb.record("26.Q9e", "B", mistakes={"divise 0 par la somme des zéros : est-ce défini ?": "A", "aucune action n'est préférée : la formule elle-même pose problème": "C"})

WB_ANSWER {"hash": "61411985752285ae4705dd2fe19cc199894c213318e9f89f3d4ccc8bd8cd466f", "id": "26.Q9a", "kind": "bool", "mistakes": {"abfa163c21a49bf6d87d6ebc6b4ee3c12944c41aad9b5bf6f96fe6f619453248": "c'est la version du livre que la fiche corrige"}}
📝 Ex 26.Q9a : réponse enregistrée (bool).
WB_ANSWER {"decimals": 2, "hash": "d86f92b68c67ba84756c7e522d067da5b40730376dbaa04c846373b4a1136934", "hash_coarse": "ab71d13ff076ca204c7a26c59d91cd768ce8f114c244985a6e7d53b61c2a85e3", "id": "26.Q9b", "kind": "float", "magnitude": -1, "mistakes": {"b9e1387eb8651bb79a40e18a9df8348bdaacadfada5865cdc2a1158c6e404294": "l'exploration peut aussi tirer l'action gloutonne"}, "sign": 1}
📝 Ex 26.Q9b : réponse enregistrée (float, 2 décimale(s)).
WB_ANSWER {"hash": "d51b786f837ed4438c01bc26fae5014772420aa3c085357411ab036e71ee9a29", "id": "26.Q9c", "kind": "bool", "mistakes": {"56cc65a1db55373050eadb2dd280d309cb8b14f7ce36ac4d1f629fc8504c3a56": "une grande température rend les probabilités presque égales"}}
📝 Ex

**26.Q10 — Table initialisée à zéro : comment la valeur remonte et converge**

In [12]:
wb.record("26.Q10a", "A", mistakes={"la mise à jour est déterministe : que vaut une cible faite de zéros ?": "B", "la mise à jour mélange des nombres nuls": "C"})
wb.record("26.Q10b", "B", mistakes={"le Q-learning met à jour un seul pas à la fois": "A", "la première action ne voit la victoire que bien plus tard": "C"})
wb.record("26.Q10c", True, mistakes={"relis le §26.6.4 : la valeur remonte grâce à la cible qui regarde l'état suivant": False})
wb.record("26.Q10d", "A", mistakes={"α = 1 remplace à chaque fois : relis l'encadré sur la convergence": "B", "une politique gloutonne ne visite pas toutes les cases": "C"})
wb.record("26.Q10e", "C", mistakes={"relis les résultats du §26.6.5 : la longueur baisse encore après 300 parties": "A", "relis les résultats du §26.6.5 : la longueur baisse encore après 3 000 parties": "B"})

WB_ANSWER {"hash": "50976916dbd2e604b8ff1cc93d599512a5f4a26e4bbe433038cc4efc57d03860", "id": "26.Q10a", "kind": "str", "mistakes": {"6a81f08a200f690c839996e67a4cce08669ca1ed1fe10ffb8829a08b2447df29": "la mise à jour est déterministe : que vaut une cible faite de zéros ?", "f2d0052a7e4cf7201274fae38774610918358e060e42ed251d18100651e005bd": "la mise à jour mélange des nombres nuls"}}
📝 Ex 26.Q10a : réponse enregistrée (str).
WB_ANSWER {"hash": "7b35eaa9edc856f30eb5e057d10cc2775b7bf3889776c9e796c38ca7cf4ff39c", "id": "26.Q10b", "kind": "str", "mistakes": {"c81b3b8efa823f1221db9e3c27533667fa7b56633a5fa73c35adad5ef35ed8d3": "le Q-learning met à jour un seul pas à la fois", "e40f1344f8be783b3ac187ed30c51390d26c2345728158f7b01ba05790931e67": "la première action ne voit la victoire que bien plus tard"}}
📝 Ex 26.Q10b : réponse enregistrée (str).
WB_ANSWER {"hash": "bbadf13a37ba997d3811cf054e32e718142a8446c5ea0f2e321f1876a194756b", "id": "26.Q10c", "kind": "bool", "mistakes": {"4b715fa81e4bdd624

**26.Q11 — SARSA contre Q-learning : l'action suivante réellement jouée**

In [13]:
wb.record("26.Q11a", "A", mistakes={"le nom liste les cinq éléments d'une mise à jour": "B", "le nom liste les cinq éléments d'une mise à jour, pas un nom propre": "C"})
wb.record("26.Q11b", "B", mistakes={"c'est la cible du Q-learning": "A", "relis la formule de SARSA": "C"})
wb.record("26.Q11c", "A", mistakes={"SARSA apprend la valeur de la politique qu'il suit": "B"})
wb.record("26.Q11d", True, mistakes={"si a' est l'action gloutonne, que vaut Q(s', a') comparé au max ?": False})
wb.record("26.Q11e", False, mistakes={"relis l'encadré « un défaut du Q-learning ? »": True})

WB_ANSWER {"hash": "3637e2d97fec7622b8befab8a057e902a90608d2b56cb802e8221411e4101130", "id": "26.Q11a", "kind": "str", "mistakes": {"9627478252f7daa494e7b94a420d625aa603b0e55c75070b4241f1c3403588c4": "le nom liste les cinq éléments d'une mise à jour", "ddcad25f56b3bdeb00c3b3eb0a5b071850dee65ba843055989056e5ef413b19b": "le nom liste les cinq éléments d'une mise à jour, pas un nom propre"}}
📝 Ex 26.Q11a : réponse enregistrée (str).
WB_ANSWER {"hash": "e5a6fe1029b97c124582c35f28a610799ba855c0178c19a85e6be249d0effcdc", "id": "26.Q11b", "kind": "str", "mistakes": {"2eae0a92c9345c241f1490809d5d43dfe9f56fd6561c18f33171dc9caca7d586": "relis la formule de SARSA", "dffb019558979af50ba97723922f080eab4800bc8919bfe6fe26039b80ed906e": "c'est la cible du Q-learning"}}
📝 Ex 26.Q11b : réponse enregistrée (str).
WB_ANSWER {"hash": "d1ea101b354ff8ead37551a8d5a5143c76d1997c768312e589b419e44627c02a", "id": "26.Q11c", "kind": "str", "mistakes": {"dbfaaea10a8859a73c6774db7640a87589ca653a49d5012699923b62189de

**26.Q12 — Tables géantes, rejeu d'expérience, AlphaGo Zero et lancer de rayons**

In [14]:
wb.record("26.Q12a", 16, mistakes={"c'est le nombre du livre, pour un plateau de 9 cases": 9})
wb.record("26.Q12b", "B", mistakes={"relis le §26.8": "A", "une table plus petite ne tiendrait pas les états": "C"})
wb.record("26.Q12c", "B", mistakes={"c'est plutôt la version du livre, pas celle de DQN": "A", "DQN garde toutes les transitions récentes": "C"})
wb.record("26.Q12d", False, mistakes={"relis le §26.10 : « Zero » veut dire zéro donnée humaine": True})
wb.record("26.Q12e", "B", mistakes={"l'adversaire n'est pas une règle de l'agent": "A", "relis le §26.10": "C"})

WB_ANSWER {"hash": "a72412a6369756edaefb2f2bf7377ec72931ae7295d61d4b49b7dfb502d35358", "id": "26.Q12a", "kind": "int", "magnitude": 1, "mistakes": {"4396028dad007ee73dbe02c3eeb2aceaa2e5edc75e16f481e5d23e2d270f1450": "c'est le nombre du livre, pour un plateau de 9 cases"}, "sign": 1}
📝 Ex 26.Q12a : réponse enregistrée (int).
WB_ANSWER {"hash": "260844a460446d5ee830a57fee786b0d32af0e72086516f13225d153e652769a", "id": "26.Q12b", "kind": "str", "mistakes": {"308270e401a5b5dd5d620a3122a9a8b255057bacc2b95eddebed67f8a124b0a6": "relis le §26.8", "360ccf44ebbff823ed2f35045036c83a1bac99d4aebe83801db30a3034bccc4d": "une table plus petite ne tiendrait pas les états"}}
📝 Ex 26.Q12b : réponse enregistrée (str).
WB_ANSWER {"hash": "a4f336960c8ea4d1ff49a597bd101e0c2757dc6bf08458abc66b6899c0041618", "id": "26.Q12c", "kind": "str", "mistakes": {"5565f74f0c52e7296e59a7a13fd948857859af9f4677a81a9f16325ab3a470c9": "DQN garde toutes les transitions récentes", "83750c0604e2e54b76cd51c1f73f4ec3552c42eb6bf30af

**26.R1 — Un réseau comme fonction qui remplace une table (le décodeur d'un VAE)**

In [15]:
wb.record("26.R1a", "C", mistakes={"les codes sont des nombres réels": "A", "les latents ne sont pas des bits": "B"})
wb.record("26.R1b", True, mistakes={"relis 25.Q11 et 25.17 : que fait un autoencodeur d'une entrée voisine de ses données ?": False})
wb.record("26.R1c", 4, mistakes={"un DQN rend une valeur par action": 1})

WB_ANSWER {"hash": "56508a6c8b5da7686ecf0a6bf22bef24a557d048b2f4fc6689062de4913d0310", "id": "26.R1a", "kind": "str", "mistakes": {"6ac190fccaf45f91198ba27a7b9c004accf070c3e41789bdc206bd634afbb956": "les codes sont des nombres réels", "aed59cd6909a8d84f5d85921fe5325f98d8f62a34dcb7ff9523d8c962200804d": "les latents ne sont pas des bits"}}
📝 Ex 26.R1a : réponse enregistrée (str).
WB_ANSWER {"hash": "0f2d4a33ab83f4fb6d0a6a1a1ae4e1217e61c921ca404eedaf843221f5953a70", "id": "26.R1b", "kind": "bool", "mistakes": {"5638671831a00bca65e3f0b283852b2afc200b7b6908fd4839859c14085f8980": "relis 25.Q11 et 25.17 : que fait un autoencodeur d'une entrée voisine de ses données ?"}}
📝 Ex 26.R1b : réponse enregistrée (bool).
WB_ANSWER {"hash": "e00f5eba01ba1af94fd2fafe497dd712f8ca139253c231e5a25a447a1cffed72", "id": "26.R1c", "kind": "int", "magnitude": 0, "mistakes": {"35721529d89d129d3ed8bdcd5a77cf6e3730ec8b02fec63951014604c3417746": "un DQN rend une valeur par action"}, "sign": 1}
📝 Ex 26.R1c : réponse 

**26.R2 — Que sauvegarder pour reprendre un entraînement ?**

In [16]:
wb.record("26.R2a", "ABCD", mistakes={"l'optimiseur a un état (les moyennes d'Adam)": "ACD",
          "sans l'état des générateurs, la suite ne sera pas la même": "ABC",
          "le code de PyTorch n'est pas un état de l'entraînement": "ABCDE",
          "il faut savoir où l'on s'est arrêté": "ABD"})
wb.record("26.R2b", "A", mistakes={"les récompenses sont un historique, pas l'état appris": "B", "l'environnement n'est pas appris": "C"})
wb.record("26.R2c", False, mistakes={"où en était ε ? et le générateur aléatoire ?": True})

WB_ANSWER {"hash": "485f5c13f92fdc3b233a0abde2528e52c7394bde1c3a2dc596d487977a3f0189", "id": "26.R2a", "kind": "str", "mistakes": {"c1a737c6b88be6912119c0ef1590328f54d6bf02618bf1c2376ff14e07ad79ab": "l'optimiseur a un état (les moyennes d'Adam)", "c2a18e3342200e800c386fb5a8c1653e4f37b2780d635ff9f929b1b4e4a7c1a4": "sans l'état des générateurs, la suite ne sera pas la même", "eed12bf0769a3cd614adfa3b15371fe0d46357f523dffa464eb882b43e2f014c": "le code de PyTorch n'est pas un état de l'entraînement", "f34abf9c99d4da5959462f52a75bd3d54f75c284062c634157ec4d546581029f": "il faut savoir où l'on s'est arrêté"}}
📝 Ex 26.R2a : réponse enregistrée (str).
WB_ANSWER {"hash": "83c4ad675676ba7a1bf9c384f23ca6bef7137ce39aeb0a64cd860941efc79608", "id": "26.R2b", "kind": "str", "mistakes": {"e10f57d0fbf478380c4cc559c269e446a117e3f697d39aa0082fb9bad171dd92": "les récompenses sont un historique, pas l'état appris", "fb748fa091608118aff6c48d7f3cb7acd364dd152a3f1a564920235ac3a5e21e": "l'environnement n'est pa

**26.R3 — Bandit à bras multiples, ε-greedy et UCB**

In [17]:
wb.record("26.R3a", "A", mistakes={"un bandit n'a pas d'état qui change": "B", "les bras sont les actions, pas des états": "C"})
wb.record("26.R3b", 3 + 0.25 * (5 - 3), decimals=2, mistakes={"le pas multiplie l'écart au nouveau gain": 3.25})
wb.record("26.R3c", "B", mistakes={"c'est la règle gloutonne": "A", "c'est l'exploration d'ε-greedy": "C"})

WB_ANSWER {"hash": "4f2ba4a66980f5e32bfa5330a9b061dc0a3dd453963cd6f3b2420222a3e95b2a", "id": "26.R3a", "kind": "str", "mistakes": {"043ff547fae91770581ce97537b8e4d8d408b682c5920568f9520e46fb8334b2": "un bandit n'a pas d'état qui change", "58db802180fe03208986ec19131e01c027ed300fa714e03d22f120dfa68e5948": "les bras sont les actions, pas des états"}}
📝 Ex 26.R3a : réponse enregistrée (str).
WB_ANSWER {"decimals": 2, "hash": "e96e7e4695bfdb5b41ea92bee6c774ef08bc5762488c3198b25fdf4f83ea957e", "hash_coarse": "a691db691e196ec31fb24588e7cfb243e862d76efca9397c0d8aea66589b7180", "id": "26.R3b", "kind": "float", "magnitude": 0, "mistakes": {"6e6eeb667299c662ebb7166fbaf06096059f8b2b07cf2bbf5c9728d232fecdf4": "le pas multiplie l'écart au nouveau gain"}, "sign": 1}
📝 Ex 26.R3b : réponse enregistrée (float, 2 décimale(s)).
WB_ANSWER {"hash": "2352c499088c07ce11ce1f3e9799165eec8b41ce0804acd3372648035d92b96c", "id": "26.R3c", "kind": "str", "mistakes": {"1f546b3188ca52481a4933ed9820e9dc408eebc6a145baf

**Ex 26.1 — Retours total et actualisé d'une partie de six coups**

In [18]:
wb.record("26.1a", sum(REWARDS_26_1))
wb.record("26.1b", sum(REWARDS_26_1[2:]), mistakes={"le TFR du coup 3 ne compte pas les récompenses d'avant": 7})
wb.record("26.1c", discounted(REWARDS_26_1, 0.5), decimals=3, mistakes={"la récompense du coup 1 n'est pas actualisée (γ⁰ = 1)": discounted(REWARDS_26_1, 0.5) * 0.5})
wb.record("26.1d", discounted(REWARDS_26_1, 0.9), decimals=3, mistakes={"la récompense du coup 1 n'est pas actualisée (γ⁰ = 1)": discounted(REWARDS_26_1, 0.9) * 0.9})
wb.record("26.1e", discounted(REWARDS_26_1[3:], 0.9), decimals=2, mistakes={"le retour du coup 4 commence à sa propre récompense": discounted(REWARDS_26_1[4:], 0.9),
          "c'est le retour total du coup 4 : actualise": 6})
wb.record("26.1f", REWARDS_26_1[0], mistakes={"γ⁰ = 1 : la récompense immédiate reste": 0})
wb.record("26.1g", True, mistakes={"G_t − G_{t+1} = r_t : quel est son signe ?": False})

WB_ANSWER {"hash": "bdef85a988a13494b8b17dd41af3cdfea1e5c9342ab7051ceddf2704fee87c16", "id": "26.1a", "kind": "int", "magnitude": 0, "sign": 1}
📝 Ex 26.1a : réponse enregistrée (int).
WB_ANSWER {"hash": "9b565d9aa5537dc758f27e8cab48f5e5e9d6bf4971e27d06795eac70baf570f4", "id": "26.1b", "kind": "int", "magnitude": 0, "mistakes": {"aac6d9a311798420387567758e600fcfff1d9d1320bc688c8559c1db6d79ee9b": "le TFR du coup 3 ne compte pas les récompenses d'avant"}, "sign": 1}
📝 Ex 26.1b : réponse enregistrée (int).
WB_ANSWER {"decimals": 3, "hash": "45786612a1b76a6a015c282a86774322002eb6dcb107219e60e31be470bdb7ce", "hash_coarse": "20fba4d7bc582d031c70e9216b2be9a6276aaed06a07802b90ff8d7c7d3b83f6", "id": "26.1c", "kind": "float", "magnitude": 0, "mistakes": {"f11a565ecee8c6353361f09673e25f36337ea2635aa1c0d3aed10cf13e8851b6": "la récompense du coup 1 n'est pas actualisée (γ⁰ = 1)"}, "sign": 1}
📝 Ex 26.1c : réponse enregistrée (float, 3 décimale(s)).
WB_ANSWER {"decimals": 3, "hash": "3c225233840361e75

**Ex 26.2 — Taille des tables : de Flippers au Go**

In [19]:
wb.record("26.2a", 2 ** 9, mistakes={"chaque tuile a deux états : 2 puissance 9, pas 9 au carré": 81, "2 × 9 ne compte pas les combinaisons": 18})
wb.record("26.2b", 2 ** 9 * 9, mistakes={"multiplie par le nombre d'actions": 512})
wb.record("26.2c", 2 ** 16 * 16, mistakes={"multiplie par le nombre d'actions": 2 ** 16, "16 actions, pas 9": 2 ** 16 * 9})
wb.record("26.2d", 4 + 4, mistakes={"les colonnes comptent aussi": 4, "les diagonales ne comptent pas": 10})
wb.record("26.2e", 3 ** 9, mistakes={"trois états par case : 3 puissance 9": 2 ** 9, "3 puissance 9, pas 9 puissance 3": 9 ** 3})
wb.record("26.2f", 3 ** 16 * 16, mistakes={"c'est le compte du livre, avec 9 colonnes": 3 ** 16 * 9, "multiplie par 16 actions": 3 ** 16})
wb.record("26.2g", math.floor(361 * math.log10(3)) + 1, mistakes={"le nombre de chiffres est la partie entière du log, plus 1": 172})

WB_ANSWER {"hash": "ad6cc83d4581bdefca7a8d08934c6d907918bc493510a9b3383524502da3ce85", "id": "26.2a", "kind": "int", "magnitude": 2, "mistakes": {"17dce6f6b6f9ad04b1721979e4e62b65e3fbd7b99d5fd782ccb454f44e527d82": "chaque tuile a deux états : 2 puissance 9, pas 9 au carré", "9886edfc24cb8725bbf8ab182d80140eb7fec9887c4063fe2095ad27938ec680": "2 × 9 ne compte pas les combinaisons"}, "sign": 1}
📝 Ex 26.2a : réponse enregistrée (int).
WB_ANSWER {"hash": "97f72136f69d5a5b3ba6e256f13b9167916a23bf420f852d2f6e1643bd1bb9bd", "id": "26.2b", "kind": "int", "magnitude": 3, "mistakes": {"26392c79ba1d33f04287ca8cce6d8df2a71998bf27db5a44c18a0ace64ec8807": "multiplie par le nombre d'actions"}, "sign": 1}
📝 Ex 26.2b : réponse enregistrée (int).
WB_ANSWER {"hash": "2daefdcf990ebdcd21848733bee85182fecdd1d14fc00467cd3f8dc69dae6532", "id": "26.2c", "kind": "int", "magnitude": 6, "mistakes": {"b82043b18446849fabf237f5d0c9629925e7b4b8c669085d4f5a1c63c4637c0b": "16 actions, pas 9", "cc0dc7097c1b20619099a6c606

**Ex 26.3 — Une mise à jour de Q-learning à la main**

In [20]:
wb.record("26.3a", TARGET_26_3, decimals=2, mistakes={"multiplie le max par γ": R_26_3 + max(NEXT_26_3),
          "le max de la ligne de s', pas la moyenne": R_26_3 + GAMMA_26_3 * sum(NEXT_26_3) / 4})
wb.record("26.3b", TARGET_26_3 - Q_26_3, decimals=2, mistakes={"δ = cible − ancienne valeur, pas l'inverse": Q_26_3 - TARGET_26_3})
wb.record("26.3c", NEW_26_3, decimals=2, mistakes={"c'est la cible : mélange-la avec l'ancienne valeur": TARGET_26_3,
          "α multiplie δ, puis on l'ajoute à l'ancienne valeur": ALPHA_26_3 * (TARGET_26_3 - Q_26_3)})
wb.record("26.3d", Q_26_3 + ALPHA_26_3 * (R_26_3 - Q_26_3), decimals=2, mistakes={"s' terminal : la cible est r seul": NEW_26_3,
          "la cible est r, mais il faut encore mélanger": R_26_3})
wb.record("26.3e", TARGET_26_3, decimals=2, mistakes={"avec α = 1, la nouvelle valeur est la cible": NEW_26_3})
wb.record("26.3f", False, mistakes={"une mise à jour ne touche qu'une case : Q(s, a)": True})

WB_ANSWER {"decimals": 2, "hash": "17beef03479d14538995aa54f386826130b69304633d3d43058b3c632bfacc3c", "hash_coarse": "761d7dcb41a5264f7f00dd78ad6948fc7e8ace1fd5ed246c38e74e015d6ef07d", "id": "26.3a", "kind": "float", "magnitude": -1, "mistakes": {"6186f3402f9a2324955c58258eb00201b971f0f33eb7b4fb6fbd95eba9dfc17d": "multiplie le max par γ", "ce4e396c00a1c4b979e8ab22dfab0224ee9ce5721c1bbce49b2509339f1aa58b": "le max de la ligne de s', pas la moyenne"}, "sign": 1}
📝 Ex 26.3a : réponse enregistrée (float, 2 décimale(s)).
WB_ANSWER {"decimals": 2, "hash": "294ec392fcf42dd8206a218256280108c9da708961c50f136f81775d8faa06e6", "hash_coarse": "6c3b64661cd6925201783c94d45fa6e2eedef22f75ba235e79383d6b9fa970bd", "id": "26.3b", "kind": "float", "magnitude": -1, "mistakes": {"b32c3a6cbdff3be5b05c93482f38ff13b13a5531c8fa3809e3ab21cb4ae0bea3": "δ = cible − ancienne valeur, pas l'inverse"}, "sign": 1}
📝 Ex 26.3b : réponse enregistrée (float, 2 décimale(s)).
WB_ANSWER {"decimals": 2, "hash": "c40bf54a8e21a

**Ex 26.4 — La même transition vue par SARSA**

In [21]:
wb.record("26.4a", TARGET_26_4, decimals=2, mistakes={"SARSA prend Q(s', a') de l'action choisie, pas le max": TARGET_26_3})
wb.record("26.4b", NEW_26_4, decimals=3, mistakes={"c'est la cible : mélange-la avec l'ancienne valeur": TARGET_26_4,
           "SARSA prend Q(s', a') de l'action choisie, pas le max": NEW_26_3})
wb.record("26.4c", NEW_26_3 - NEW_26_4, decimals=3, mistakes={"Q-learning moins SARSA, pas l'inverse": NEW_26_4 - NEW_26_3})
wb.record("26.4d", True, mistakes={"Q(s', 1) vaut 0,6, le max de la ligne": False})
wb.record("26.4e", "B", mistakes={"le Q-learning apprend la valeur de la politique gloutonne, pas de celle qu'il suit": "A"})

WB_ANSWER {"decimals": 2, "hash": "1a2a2b0f74e6d0f9a35e105f144bdee8d2f272aebe75b8bd8758a6159c4084cb", "hash_coarse": "fa7a0a6bd6c5e0d40a596b58610d3923eda94260c1bedf2ebceecab7f1544af8", "id": "26.4a", "kind": "float", "magnitude": -1, "mistakes": {"65f70807208ae8161071531938d9e4632b1f0de9a7e8eb930b453566a571873c": "SARSA prend Q(s', a') de l'action choisie, pas le max"}, "sign": 1}
📝 Ex 26.4a : réponse enregistrée (float, 2 décimale(s)).
WB_ANSWER {"alt_coarse": ["3c541efce94aa6b63e7b20adfafb02fd314ecd306138ebd441ed7b9eeb701d7a"], "decimals": 3, "hash": "1f2fcd27071d89c37070720843afd3b9772c4325fbe42144c944784d3363941e", "hash_coarse": "15ea303da1e5f788ab86a0e584568ba1f40b1ba13277ec59422a9a748d952ece", "id": "26.4b", "kind": "float", "magnitude": -1, "mistakes": {"139f305849b3768cfd8c12cfc0c1579c91ceeca76f1f8af8c703749ea2a5c3ae": "SARSA prend Q(s', a') de l'action choisie, pas le max", "910e9d38af1624aa758bc2211d4d1dc440f0e5b2b0f5b6a31eddb17099757fef": "c'est la cible : mélange-la avec l

**Ex 26.5 — Probabilités d'action : ε-greedy (standard et version du livre) et softmax**

In [22]:
wb.record("26.5a", 1 - EPS_26_5 + EPS_26_5 / 4, decimals=2, mistakes={"l'exploration standard peut aussi tirer l'action gloutonne": 1 - EPS_26_5})
wb.record("26.5b", EPS_26_5 / 4, decimals=2, mistakes={"l'action 0 n'est jouée qu'en exploration : ε divisé par le nombre d'actions": EPS_26_5})
wb.record("26.5c", 1 - EPS_26_5, decimals=2, mistakes={"dans la version du livre, l'exploration ne tire jamais la gloutonne": 1 - EPS_26_5 + EPS_26_5 / 4})
wb.record("26.5d", EPS_26_5 / 3, decimals=3, mistakes={"le livre tire parmi les trois autres actions": EPS_26_5 / 4})
wb.record("26.5e", softmax_prob(Q_26_5, 1, 2), decimals=3, mistakes={"c'est la « softmax » du livre : la vraie passe par l'exponentielle": 4 / 7})
wb.record("26.5f", softmax_prob(Q_26_5, 2, 2), decimals=3, mistakes={"divise les valeurs par τ = 2 avant l'exponentielle": softmax_prob(Q_26_5, 1, 2),
          "on divise par τ, on ne multiplie pas": softmax_prob(Q_26_5, 0.5, 2)})
wb.record("26.5g", False, mistakes={"e^(Q+10) = e^10 × e^Q : le facteur se simplifie": True})
wb.record("26.5h", 4 / 7, decimals=3, mistakes={"c'est la vraie softmax : le livre divise les valeurs par leur somme": softmax_prob(Q_26_5, 1, 2)})

WB_ANSWER {"decimals": 2, "hash": "f7c3b3b1fc789a6c2964e0a5c24d6b3388ee9b97434fc0c6379e1661ea2b21bc", "hash_coarse": "e4f6ab7e571f2cf944df7c2a0f3728d5ceadf12508d27471887ea45f503c73f7", "id": "26.5a", "kind": "float", "magnitude": -1, "mistakes": {"81baebac60cb006bfd94c5b015839caca3c156c4619d2936ff3d03edd7a4ea29": "l'exploration standard peut aussi tirer l'action gloutonne"}, "sign": 1}
📝 Ex 26.5a : réponse enregistrée (float, 2 décimale(s)).
WB_ANSWER {"decimals": 2, "hash": "3093dd3b85dbb854a3fbab733173594f1195111db059222533e9ffddc8f4cbb0", "hash_coarse": "6d83eba4b7fab661dd182e1864bfb5d6a0e85d7776e38871eb04bcce0f7befe8", "id": "26.5b", "kind": "float", "magnitude": -2, "mistakes": {"cfc761b24fc312c26b77841d32911f78f5fc2d4828a296ee2fc82fbaa845e24a": "l'action 0 n'est jouée qu'en exploration : ε divisé par le nombre d'actions"}, "sign": 1}
📝 Ex 26.5b : réponse enregistrée (float, 2 décimale(s)).
WB_ANSWER {"decimals": 2, "hash": "93c87e3f3039d8bcf4aaff7664d49efcddb317839040da32d8926036

**Ex 26.6 — Regarder la valeur remonter une chaîne d'états**

In [23]:
wb.record("26.6a", round(chain_26_6(1.0, 1)[2]))
wb.record("26.6b", chain_26_6(1.0, 2)[1], decimals=2, mistakes={"le pas de 1 à 2 ne rapporte rien : seule la valeur actualisée de l'état 2 compte": 1.0})
wb.record("26.6c", chain_26_6(1.0, 3)[0], decimals=2, mistakes={"compte les actualisations entre l'état 0 et la récompense": 0.9})
wb.record("26.6d", 3, mistakes={"les mises à jour suivent l'ordre des pas : l'état 0 est mis à jour avant que 1 ait une valeur": 1,
          "à cet épisode, l'état 1 avait-il déjà une valeur quand l'état 0 a été mis à jour ?": 2})
wb.record("26.6e", chain_26_6(0.5, 2)[1], decimals=3, mistakes={"à l'épisode 2, l'état 1 voit la valeur de l'état 2 d'avant sa propre mise à jour": 0.5 * 0.9 * 0.75,
          "α = 0,5 : on ne garde que la moitié de l'écart": 0.9 * 0.5})
wb.record("26.6f", chain_26_6(0.5, 3)[2], decimals=3, mistakes={"compte les mises à jour de l'état 2 pendant les trois épisodes": 0.75})
wb.record("26.6g", 0.9 ** 2, decimals=2, mistakes={"α ne change pas la limite, seulement la vitesse": 0.5 * 0.81,
          "compte les actualisations entre l'état 0 et la récompense": 0.9})
wb.record("26.6h", 10, mistakes={"refais le raisonnement de d) : à quel épisode chaque état reçoit-il sa première valeur ?": 11,
          "compte les états non terminaux de la chaîne": 9})

WB_ANSWER {"hash": "e25a73150a8cc2c712ccd33a091c458d61ec01183d40dfbb4022bb25a670c052", "id": "26.6a", "kind": "int", "magnitude": 0, "sign": 1}
📝 Ex 26.6a : réponse enregistrée (int).
WB_ANSWER {"decimals": 2, "hash": "b7dc94dbdc5a82a44356d9b560551e002c85e4d4318718852a2ce7326f4d68f8", "hash_coarse": "b88d269a81234d4f63066af7ec8cb8a08d0b06e516451f12cf247dea44b972fe", "id": "26.6b", "kind": "float", "magnitude": -1, "mistakes": {"377dc9558db258a5795d5a1100c0a307cadc729574d8896886be387e0b1c3b6e": "le pas de 1 à 2 ne rapporte rien : seule la valeur actualisée de l'état 2 compte"}, "sign": 1}
📝 Ex 26.6b : réponse enregistrée (float, 2 décimale(s)).
WB_ANSWER {"decimals": 2, "hash": "ad6d18eda31093b18d6a18f750306eafbe8985d71d8b8be8278afc29370ed918", "hash_coarse": "c3a7ebaaa0665a62be0cf660db03265c0e44982cbd2c099094c5757e2d6419a5", "id": "26.6c", "kind": "float", "magnitude": -1, "mistakes": {"177f91e6763a53adc09a024e604db8f6aa6cfb797d8c180735b45f21e33d9f9f": "compte les actualisations entre 

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu écrire et appliquer à la main les mises à jour du Q-learning et de SARSA, état terminal compris ?
2. Sais-tu expliquer ce que règlent $\alpha$, $\gamma$ et $\varepsilon$, et ce qui se passe quand on les choisit mal ?
3. Sais-tu dire pourquoi une table ne passe pas à l'échelle, et ce que DQN y ajoute ?

**Pour aller plus loin** : les chapitres 2, 3 et 6 de Sutton et Barto (2018), cités dans la fiche. La suite : le ch. 27, les GAN.

*Les exercices de code 26.13 à 26.38 (parties A à F : la boucle et Flippers, ta librairie `rl.py`, le Q-learning, SARSA, le rejeu et les récompenses, le morpion, CartPole et l'auto-apprentissage) seront ajoutés à la prochaine session de génération ; relance alors `python tools/start_chapter.py 26` pour obtenir le notebook complet.*